In [6]:
!pip install pymongo
!pip install pandas

In [7]:
from pymongo import MongoClient, InsertOne, UpdateOne, DeleteOne

client = MongoClient('mongodb://localhost:27017/')
db = client['university_db']
courses_collection = db['courses']

operations = [
    InsertOne({'course': 'Math 101', 'enrollments': 30, 'department': 'Mathematics'}),
    InsertOne({'course': 'CS 102', 'enrollments': 25, 'department': 'Computer Science'}),
    InsertOne({'course': 'History 201', 'enrollments': 20, 'department': 'History'}),
    InsertOne({'course': 'Physics 202', 'enrollments': 15, 'department': 'Physics'}),
    InsertOne({'course': 'Chemistry 301', 'enrollments': 28, 'department': 'Chemistry'}),
    InsertOne({'course': 'Biology 204', 'enrollments': 22, 'department': 'Biology'}),
    InsertOne({'course': 'Economics 210', 'enrollments': 19, 'department': 'Economics'}),
    InsertOne({'course': 'English 110', 'enrollments': 26, 'department': 'English Literature'})
]

courses_collection.delete_many({})

result = courses_collection.bulk_write(operations)
print("Kursus berhasil dimasukkan.")
print("Jumlah data yang dimasukkan:", len(result.inserted_ids) if hasattr(result, 'inserted_ids') else len(operations))

print("\nDaftar kursus dengan pendaftaran:")
for course in courses_collection.find():
    print(f"Course: {course['course']}, "
          f"Department: {course['department']}, "
          f"Enrollments: {course['enrollments']}")


Kursus berhasil dimasukkan.
Jumlah data yang dimasukkan: 8

Daftar kursus dengan pendaftaran:
Course: Math 101, Department: Mathematics, Enrollments: 30
Course: CS 102, Department: Computer Science, Enrollments: 25
Course: History 201, Department: History, Enrollments: 20
Course: Physics 202, Department: Physics, Enrollments: 15
Course: Chemistry 301, Department: Chemistry, Enrollments: 28
Course: Biology 204, Department: Biology, Enrollments: 22
Course: Economics 210, Department: Economics, Enrollments: 19
Course: English 110, Department: English Literature, Enrollments: 26


In [8]:
print("Mata Kuliah dengan Jumlah Mahasiswa > 20:")
for course in courses_collection.find({'enrollments': {'$gt': 20}}):
    print(f"Nama Mata Kuliah: {course['course']}, "
          f"Departemen: {course['department']}, "
          f"Jumlah Mahasiswa: {course['enrollments']}")

print("\nMata Kuliah dari Departemen Computer Science atau Mathematics:")
for course in courses_collection.find({'department': {'$in': ['Computer Science', 'Mathematics']}}):
    print(f"Nama Mata Kuliah: {course['course']}, "
          f"Departemen: {course['department']}, "
          f"Jumlah Mahasiswa: {course['enrollments']}")


Mata Kuliah dengan Jumlah Mahasiswa > 20:
Nama Mata Kuliah: Math 101, Departemen: Mathematics, Jumlah Mahasiswa: 30
Nama Mata Kuliah: CS 102, Departemen: Computer Science, Jumlah Mahasiswa: 25
Nama Mata Kuliah: Chemistry 301, Departemen: Chemistry, Jumlah Mahasiswa: 28
Nama Mata Kuliah: Biology 204, Departemen: Biology, Jumlah Mahasiswa: 22
Nama Mata Kuliah: English 110, Departemen: English Literature, Jumlah Mahasiswa: 26

Mata Kuliah dari Departemen Computer Science atau Mathematics:
Nama Mata Kuliah: Math 101, Departemen: Mathematics, Jumlah Mahasiswa: 30
Nama Mata Kuliah: CS 102, Departemen: Computer Science, Jumlah Mahasiswa: 25


In [9]:
pipeline = [
    {
        '$group': {
            '_id': '$department',                      # mengelompokkan data berdasarkan field 'department'
            'rata_rata_mahasiswa': {'$avg': '$enrollments'}  # menghitung rata-rata dari field 'enrollments'
        }
    }
]

print("Rata-rata Jumlah Mahasiswa per Departemen:")
for result in courses_collection.aggregate(pipeline):
    print(f"Departemen: {result['_id']}, Rata-rata Mahasiswa: {result['rata_rata_mahasiswa']:.2f}")

#
pipeline = [
    {
        '$group': {
            '_id': '$department',                      # mengelompokkan data berdasarkan departemen
            'jumlah_maksimum': {'$max': '$enrollments'}    # mengambil nilai maksimum dari field 'enrollments'
        }
    }
]

print("\nJumlah Mahasiswa Maksimum per Departemen:")
for result in courses_collection.aggregate(pipeline):
    print(f"Departemen: {result['_id']}, Jumlah Maksimum: {result['jumlah_maksimum']}")


Rata-rata Jumlah Mahasiswa per Departemen:
Departemen: Mathematics, Rata-rata Mahasiswa: 30.00
Departemen: English Literature, Rata-rata Mahasiswa: 26.00
Departemen: Physics, Rata-rata Mahasiswa: 15.00
Departemen: Computer Science, Rata-rata Mahasiswa: 25.00
Departemen: Biology, Rata-rata Mahasiswa: 22.00
Departemen: Economics, Rata-rata Mahasiswa: 19.00
Departemen: Chemistry, Rata-rata Mahasiswa: 28.00
Departemen: History, Rata-rata Mahasiswa: 20.00

Jumlah Mahasiswa Maksimum per Departemen:
Departemen: Mathematics, Jumlah Maksimum: 30
Departemen: English Literature, Jumlah Maksimum: 26
Departemen: Physics, Jumlah Maksimum: 15
Departemen: Computer Science, Jumlah Maksimum: 25
Departemen: Biology, Jumlah Maksimum: 22
Departemen: Economics, Jumlah Maksimum: 19
Departemen: Chemistry, Jumlah Maksimum: 28
Departemen: History, Jumlah Maksimum: 20


In [10]:
# 
pipeline = [
    {
        '$project': {
            'course_name': '$course',             # mengubah nama field 'course' menjadi 'course_name'
            'department_name': '$department',     # mengubah nama field 'department' menjadi 'department_name'
            'enrollments': 1                      # menampilkan field 'enrollments' apa adanya
        }
    }
]

print("Data Mata Kuliah dengan Penamaan Field Baru:")
for result in courses_collection.aggregate(pipeline):
    print(f"Nama Mata Kuliah: {result['course_name']}, "
          f"Departemen: {result['department_name']}, "
          f"Jumlah Mahasiswa: {result['enrollments']}")

#
pipeline = [
    {
        '$addFields': {
            'enrollment_category': {              # membuat field baru bernama 'enrollment_category'
                '$cond': {                        # menggunakan operator $cond (kondisi if-else)
                    'if': {'$gt': ['$enrollments', 20]},  # jika enrollments > 20
                    'then': 'high',                        # maka kategorinya 'high'
                    'else': 'low'                          # selain itu kategorinya 'low'
                }
            }
        }
    }
]

print("\nData Mata Kuliah dengan Kategori Jumlah Mahasiswa:")
for result in courses_collection.aggregate(pipeline):
    print(f"Nama Mata Kuliah: {result['course']}, "
          f"Departemen: {result['department']}, "
          f"Jumlah Mahasiswa: {result['enrollments']}, "
          f"Kategori: {result['enrollment_category']}")


Data Mata Kuliah dengan Penamaan Field Baru:
Nama Mata Kuliah: Math 101, Departemen: Mathematics, Jumlah Mahasiswa: 30
Nama Mata Kuliah: CS 102, Departemen: Computer Science, Jumlah Mahasiswa: 25
Nama Mata Kuliah: History 201, Departemen: History, Jumlah Mahasiswa: 20
Nama Mata Kuliah: Physics 202, Departemen: Physics, Jumlah Mahasiswa: 15
Nama Mata Kuliah: Chemistry 301, Departemen: Chemistry, Jumlah Mahasiswa: 28
Nama Mata Kuliah: Biology 204, Departemen: Biology, Jumlah Mahasiswa: 22
Nama Mata Kuliah: Economics 210, Departemen: Economics, Jumlah Mahasiswa: 19
Nama Mata Kuliah: English 110, Departemen: English Literature, Jumlah Mahasiswa: 26

Data Mata Kuliah dengan Kategori Jumlah Mahasiswa:
Nama Mata Kuliah: Math 101, Departemen: Mathematics, Jumlah Mahasiswa: 30, Kategori: high
Nama Mata Kuliah: CS 102, Departemen: Computer Science, Jumlah Mahasiswa: 25, Kategori: high
Nama Mata Kuliah: History 201, Departemen: History, Jumlah Mahasiswa: 20, Kategori: low
Nama Mata Kuliah: Physic

In [13]:
from pymongo import MongoClient, InsertOne

#menghubungkan ke MongoDB lokal dan memilih database + koleksi
client = MongoClient("mongodb://localhost:27017/")
db = client["university_db"]
courses_collection = db["courses"]
students_collection = db["students"]

#HOMEWORK 1
print("Tugas 1: jumlah mata kuliah per departemen")
pipeline_hw1 = [
    {
        "$group": {
            "_id": "$department",          # kelompokkan berdasarkan 'department'
            "jumlah_matakuliah": {"$sum": 1}  # hitung total dokumen per kelompok
        }
    }
]

results_hw1 = list(courses_collection.aggregate(pipeline_hw1))
if not results_hw1:
    print("Tidak ada data pada koleksi 'courses'.")
else:
    for r in results_hw1:
        print(f"Departemen: {r['_id']}, Jumlah Mata Kuliah: {r['jumlah_matakuliah']}")
print()

#HOMEWORK 2
print("Tugas 2: course (enrollments > 25) di departemen computer science")
pipeline_hw2 = [
    {
        "$match": {
            "department": "Computer Science",
            "enrollments": {"$gt": 25}     
        }
    },
    {
        "$group": {
            "_id": "$department",
            "total_kursus": {"$sum": 1},
            "rata_enrollments": {"$avg": "$enrollments"}
        }
    }
]

results_hw2 = list(courses_collection.aggregate(pipeline_hw2))
if not results_hw2:
    print("Tidak ada course yang memenuhi syarat (enrollments > 25) pada 'Computer Science'.")
else:
    for r in results_hw2:
        print(f"Departemen: {r['_id']}, Total Mata Kuliah: {r['total_kursus']}, "
              f"Rata-rata Mahasiswa: {r['rata_enrollments']:.2f}")
print()

# HOMEWORK 3
#(3.1) reset & isi 'students' agar konsisten dan tidak dobel
students_collection.delete_many({})  #hindari duplikasi saat run ulang

ops_students = [
    InsertOne({"student_name": "Lumine",         "course": "Math 101",      "grade": "A"}),
    InsertOne({"student_name": "Aether",         "course": "CS 102",        "grade": "B+"}),
    InsertOne({"student_name": "Xiao",           "course": "Math 101",      "grade": "A-"}),
    InsertOne({"student_name": "Zhongli",        "course": "History 201",   "grade": "A"}),
    InsertOne({"student_name": "Hu Tao",         "course": "Physics 202",   "grade": "B"}),
    InsertOne({"student_name": "Klee",           "course": "Chemistry 301", "grade": "A"}),
    InsertOne({"student_name": "Raiden Shogun",  "course": "English 110",   "grade": "A+"}),
    InsertOne({"student_name": "Kazuha",         "course": "Economics 210", "grade": "B+"}),
    InsertOne({"student_name": "Ayaka",          "course": "Biology 204",   "grade": "A"}),
    InsertOne({"student_name": "Childe",         "course": "Math 101",      "grade": "B"}),
]
students_collection.bulk_write(ops_students)
print("Tugas 3: data 'students' berhasil diisi.\n")

#(3.2) $lookup + $project: gabungkan courses - students
pipeline_hw3 = [
    {
        "$lookup": {
            "from": "students",           # koleksi yang akan digabung
            "localField": "course",       # field di 'courses'
            "foreignField": "course",     # field padanan di 'students'
            "as": "student_details"       # hasil join dimasukkan ke array 'student_details'
        }
    },
    {
        "$project": {
            "_id": 0,
            "course": 1,
            "department": 1,
            "enrollments": 1,
            "student_details.student_name": 1,
            "student_details.grade": 1
        }
    }
]

print("Tugas 3: hasil join courses - students")
for doc in courses_collection.aggregate(pipeline_hw3):
    print(f"Mata Kuliah: {doc['course']} ({doc['department']})")
    print(f"Jumlah Mahasiswa Terdaftar (enrollments): {doc['enrollments']}")
    if doc.get("student_details"):
        print("Daftar Mahasiswa (hasil join):")
        for s in doc["student_details"]:
            #beberapa dokumen mungkin tidak memiliki grade (jaga-jaga), gunakan get()
            print(f" - {s.get('student_name', '(tanpa nama)')} (Nilai: {s.get('grade', '-')})")
    else:
        print("Tidak ada mahasiswa yang terdata pada koleksi 'students' untuk course ini.")
    print("-" * 60)


Tugas 1: jumlah mata kuliah per departemen
Departemen: Mathematics, Jumlah Mata Kuliah: 1
Departemen: English Literature, Jumlah Mata Kuliah: 1
Departemen: Physics, Jumlah Mata Kuliah: 1
Departemen: Computer Science, Jumlah Mata Kuliah: 1
Departemen: Biology, Jumlah Mata Kuliah: 1
Departemen: Economics, Jumlah Mata Kuliah: 1
Departemen: Chemistry, Jumlah Mata Kuliah: 1
Departemen: History, Jumlah Mata Kuliah: 1

Tugas 2: course (enrollments > 25) di departemen computer science
Tidak ada course yang memenuhi syarat (enrollments > 25) pada 'Computer Science'.

Tugas 3: data 'students' berhasil diisi.

Tugas 3: hasil join courses - students
Mata Kuliah: Math 101 (Mathematics)
Jumlah Mahasiswa Terdaftar (enrollments): 30
Daftar Mahasiswa (hasil join):
 - Lumine (Nilai: A)
 - Xiao (Nilai: A-)
 - Childe (Nilai: B)
------------------------------------------------------------
Mata Kuliah: CS 102 (Computer Science)
Jumlah Mahasiswa Terdaftar (enrollments): 25
Daftar Mahasiswa (hasil join):
 - A